In [18]:
import pandas as pd
from pipepline_utils import load_all_zips, AccelPipeline
from sklearn.pipeline import  Pipeline

from modeling_utils  import ActivityModeler
data = load_all_zips('/home/rajesh/work/acclerometer_project/zip_data')
pipeline = AccelPipeline(data)
pipeline.convert_to_gravity()
pipeline.calc_odba()
pipeline.calc_vedba()




--- Converting to Gravity Units & Calculating ENMO ---
--- Calculating ODBA ---
--- Calculating VeDBA ---


,UTCdate,local_ts,x,y,z,subject,behavioral_category,behavior_type,x_g,y_g,z_g,mag,enmo,odba,vedba
0,2025-07-22 22:21:35,2025-07-22 17:21:35,-16536.0,-2196.0,-1392.0,500,Resting,START,-1.009277,-0.134033,-0.084961,1.021677,0.021677,0.049805,0.034024
1,2025-07-22 22:21:35,2025-07-22 17:21:35,-16536.0,-2196.0,-1392.0,500,Resting,START,-1.009277,-0.134033,-0.084961,1.021677,0.021677,0.024902,0.017012
2,2025-07-22 22:21:35,2025-07-22 17:21:35,-16608.0,-1584.0,-1968.0,500,Resting,START,-1.013672,-0.096680,-0.120117,1.025332,0.025332,0.042236,0.028102
3,2025-07-22 22:21:35,2025-07-22 17:21:35,-16476.0,-972.0,-1548.0,500,Resting,START,-1.005615,-0.059326,-0.094482,1.011785,0.011785,0.051819,0.036953
4,2025-07-22 22:21:35,2025-07-22 17:21:35,-16608.0,-2292.0,-1104.0,500,Resting,START,-1.013672,-0.139893,-0.067383,1.025496,0.025496,0.037646,0.026934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2450928,2025-07-30 17:28:15,2025-07-30 12:28:15,-15156.0,-3036.0,-5628.0,1144,Resting,START,-0.925049,-0.185303,-0.343506,1.004016,0.004016,0.037231,0.025439
2450929,2025-07-30 17:28:15,2025-07-30 12:28:15,-14628.0,-3252.0,-5868.0,1144,Resting,START,-0.892822,-0.198486,-0.358154,0.982244,0.000000,0.031913,0.021805
2450930,2025-07-30 17:28:15,2025-07-30 12:28:15,-15036.0,-3036.0,-5652.0,1144,Resting,START,-0.917725,-0.185303,-0.344971,0.997778,0.000000,0.029938,0.022365
2450931,2025-07-30 17:28:15,2025-07-30 12:28:15,-15180.0,-2916.0,-5748.0,1144,Resting,START,-0.926514,-0.177979,-0.350830,1.006571,0.006571,0.031169,0.023724


In [ ]:
def run_classical_pipeline(pipeline, time_steps):
    import ast
    from sklearn.preprocessing import LabelEncoder
    from sklearn.linear_model import LogisticRegression
    import xgboost as xgb  # Import XGBoost
    import ast
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.linear_model import LogisticRegression
    temp_results = []
    resampled_map = {}

    for t in time_steps:
        df_ready = pipeline.resample_data(interval_seconds=t)

        # skip if no data or no target
        if df_ready is None or df_ready.empty or 'behavioral_category' not in df_ready.columns:
            continue
        df_ready= df_ready.sample(frac=0.01)
        modeler = ActivityModeler(df_ready, target_col='behavioral_category')
        results = modeler.run_optuna_experiments()

        # support (df, path) return or df only
        if isinstance(results, tuple):
            results_df = results[0]
        else:
            results_df = results

        if results_df is None or results_df.empty:
            continue

        results_df = results_df.copy()
        results_df['time'] = t
        temp_results.append(results_df)
        resampled_map[t] = df_ready

    if not temp_results:
        None

    final_df = pd.concat(temp_results, ignore_index=True)

    # pick best by highest F1
    best_row = final_df.sort_values('F1_Score', ascending=False).iloc[0]

    best_time = best_row['time']
    best_algo = best_row['Algorithm']

    # parse params and features (they may be stored as strings)
    try:
        best_params = ast.literal_eval(best_row['Best_Params'])
    except Exception:
        best_params = {}

    try:
        best_features = ast.literal_eval(best_row['Features_Used'])
    except Exception:
        best_features = best_row['Features_Used']

    # get the resampled dataframe for the best time
    df_best = resampled_map.get(best_time)
    if df_best is None:
        None

    X_full = df_best[best_features]
    y_full = LabelEncoder().fit_transform(df_best['behavioral_category'])

    # reconstruct model
    if best_algo == "XGBoost":
        model = xgb.XGBClassifier(**best_params, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
    elif best_algo == "RandomForest":
        model = RandomForestClassifier(**best_params, random_state=42)
    elif best_algo == "SVM (RBF)":
        model = SVC(**best_params, kernel='rbf', random_state=42)
    elif best_algo == "KNN":
        model = KNeighborsClassifier(**best_params)
    elif best_algo == "LogisticReg":
        model = LogisticRegression(**best_params, solver='lbfgs', max_iter=2000, random_state=42)
    else:
        None

    pipeline_obj = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipeline_obj.fit(X_full, y_full)

    return {
        'model': pipeline_obj,
        'time': best_time,
        'algorithm': best_algo,
        'best_params': best_params,
        'features': best_features,
        'f1_score': best_row.get('F1_Score', None)
    }

time_steps=[10,15,20]
modal_dict=run_classical_pipeline(pipeline, time_steps)
test_df= pd.read_excel('/home/rajesh/work/acclerometer_project/data/30July25/Processed Files/processed 500_AED4_30July25_700_900.xlsx')
test_pipeline = AccelPipeline(test_df)
test_pipeline.convert_to_gravity()
test_pipeline.calc_odba()
test_pipeline.calc_vedba()
data= test_pipeline.resample_data(model_dict['time'])
data_test = data[model_dict['features']]

y_pred = model_dict['model'].predict(data_test)
y_true=data['behavioral_category']
data['predicted']=y_pred
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
f1        = f1_score(y_true, y_pred, average='weighted')



--- Resampling data to 10 second windows ---
Starting Optuna Tuning on 203 rows.
Optimizing 8 Feature Sets x 5 Models...
------------------------------------------------------------
 Tuning XGBoost | Features: Indiv: Raw Accel...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [18:52:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Tuning RandomForest | Features: Indiv: Raw Accel...
 Tuning SVM (RBF) | Features: Indiv: Raw Accel...
 Tuning KNN | Features: Indiv: Raw Accel...
 Tuning LogisticReg | Features: Indiv: Raw Accel...
 Tuning XGBoost | Features: Indiv: ODBA...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [18:53:03] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Tuning RandomForest | Features: Indiv: ODBA...
 Tuning SVM (RBF) | Features: Indiv: ODBA...
 Tuning KNN | Features: Indiv: ODBA...
 Tuning LogisticReg | Features: Indiv: ODBA...
 Tuning XGBoost | Features: Indiv: VeDBA...


...

/home/rajesh/work/basic_stat/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [19:09:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Tuning RandomForest | Features: Seq: Raw Accel + ODBA + VeDBA...
 Tuning SVM (RBF) | Features: Seq: Raw Accel + ODBA + VeDBA...
 Tuning KNN | Features: Seq: Raw Accel + ODBA + VeDBA...
 Tuning LogisticReg | Features: Seq: Raw Accel + ODBA + VeDBA...
 Tuning XGBoost | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [19:09:33] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Tuning RandomForest | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
 Tuning SVM (RBF) | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
 Tuning KNN | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
 Tuning LogisticReg | Features: Seq: Raw Accel + ODBA + VeDBA + Magnitude...
--- Converting to Gravity Units & Calculating ENMO ---
--- Calculating ODBA ---
--- Calculating VeDBA ---
--- Resampling data to 100 second windows ---


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
def run_timeseries_pipeline(pipeline, time_steps):
    import ast
    import torch
    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import LabelEncoder
    from torch.utils.data import DataLoader

    # 🔽 IMPORT from your single-file deep modeler
    from time_series_utils import (
        DeepActivityModeler,
        WindowedTimeSeriesDataset,
        LSTMClassifier,
        CNN1DClassifier
    )

    temp_results = []
    resampled_map = {}

    # ======================================================
    # 1. TRAIN ACROSS TIME WINDOWS
    # ======================================================
    for t in time_steps:
        print(f"\n⏱ Running time window = {t}s")

        df_ready = pipeline.resample_data(interval_seconds=t)

        if df_ready is None or df_ready.empty:
            continue
        if 'behavioral_category' not in df_ready.columns:
            continue

        # Optional subsampling (same as classical)
        df_ready = df_ready.sample(frac=0.01, random_state=42)

        modeler = DeepActivityModeler(
            df_ready,
            target_col='behavioral_category'
        )

        results_df = modeler.run_optuna_experiments(
            window_size=t,
            n_trials=20
        )

        if results_df is None or results_df.empty:
            continue

        results_df = results_df.copy()
        results_df["time"] = t

        temp_results.append(results_df)
        resampled_map[t] = df_ready

    if not temp_results:
        raise RuntimeError("No valid time-series models trained.")

    final_df = pd.concat(temp_results, ignore_index=True)

    # ======================================================
    # 2. SELECT BEST MODEL (GLOBAL)
    # ======================================================
    best_row = final_df.sort_values("Best_F1", ascending=False).iloc[0]

    best_time = best_row["time"]
    best_model_name = best_row["Model"]
    best_features = ast.literal_eval(best_row["Features_Used"])
    best_params = ast.literal_eval(best_row["Best_Params"])

    df_best = resampled_map[best_time]

    # ======================================================
    # 3. PREPARE FULL DATA
    # ======================================================
    X = df_best[best_features].values
    le = LabelEncoder()
    y = le.fit_transform(df_best["behavioral_category"])
    n_classes = len(np.unique(y))

    device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = WindowedTimeSeriesDataset(
        X, y, window_size=best_time
    )

    loader = DataLoader(
        dataset,
        batch_size=best_params.get("batch_size", 64),
        shuffle=True
    )

    # ======================================================
    # 4. REBUILD BEST MODEL
    # ======================================================
    if best_model_name == "LSTM":
        model = LSTMClassifier(
            n_features=X.shape[1],
            hidden_dim=best_params["hidden_dim"],
            n_layers=best_params["n_layers"],
            n_classes=n_classes,
            dropout=best_params["dropout"]
        )

    elif best_model_name == "CNN":
        model = CNN1DClassifier(
            n_features=X.shape[1],
            n_filters=best_params["n_filters"],
            kernel_size=best_params["kernel_size"],
            n_classes=n_classes
        )

    else:
        raise ValueError(f"Unknown model {best_model_name}")

    model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_params["lr"]
    )

    loss_fn = torch.nn.CrossEntropyLoss()

    # ======================================================
    # 5. FINAL TRAINING (FULL DATA)
    # ======================================================
    model.train()
    for _ in range(20):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

    return {
        "model": model,
        "label_encoder": le,
        "time": best_time,
        "model_type": best_model_name,
        "features": best_features,
        "best_params": best_params,
        "f1_score": best_row["Best_F1"]
    }
time_steps = [10, 15, 20]

model_dict = run_timeseries_pipeline(
    pipeline=pipeline,
    time_steps=time_steps
)




⏱ Running time window = 10s
--- Resampling data to 10 second windows ---
🔍 LSTM | Seq: Raw Accel | window=10


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.13464832911925095 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.048029570142568145 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.41282260596818743 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout afte

🔍 CNN | Seq: Raw Accel | window=10
🔍 LSTM | Seq: Raw Accel + ODBA | window=10


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.24682162738393776 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3239984484286648 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.306860407546387 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after al

🔍 CNN | Seq: Raw Accel + ODBA | window=10
🔍 LSTM | Seq: Raw Accel + ODBA + VeDBA | window=10


...

🔍 CNN | Seq: Raw Accel + ODBA + VeDBA | window=20
🔍 LSTM | Seq: Raw Accel + ODBA + VeDBA + Magnitude | window=20


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.0605487893555306 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.10425049192925673 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2722859605553827 and num_layers=1
  warnings.warn(
/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after a

🔍 CNN | Seq: Raw Accel + ODBA + VeDBA + Magnitude | window=20


/home/rajesh/work/basic_stat/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.09342038296786896 and num_layers=1
  warnings.warn(


ModuleNotFoundError: No module named 'deep_timeseries_activity_modeler'

In [91]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from time_series_utils import WindowedTimeSeriesDataset
from torch.utils.data import DataLoader

# Load test data
test_df = pd.read_excel(
    "/home/rajesh/work/acclerometer_project/data/30July25/Processed Files/"
    "processed 500_AED4_30July25_700_900.xlsx"
)

test_pipeline = AccelPipeline(test_df)
test_pipeline.convert_to_gravity()
test_pipeline.calc_odba()
test_pipeline.calc_vedba()

data = test_pipeline.resample_data(
    interval_seconds=model_dict["time"]
)

X_test = data[model_dict["features"]].values
y_true = data["behavioral_category"].values

dataset = WindowedTimeSeriesDataset(
    X_test,
    model_dict["label_encoder"].transform(y_true),
    window_size=model_dict["time"]
)

loader = DataLoader(dataset, batch_size=64, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model_dict["model"]
model.eval()

y_pred = []

with torch.no_grad():
    for xb, _ in loader:
        xb = xb.to(device)
        preds = model(xb).argmax(1).cpu().numpy()
        y_pred.extend(preds)

y_pred = model_dict["label_encoder"].inverse_transform(y_pred)

data = data.iloc[model_dict["time"]:]  # align with windows
data["predicted"] = y_pred

accuracy  = accuracy_score(data["behavioral_category"], y_pred)
precision = precision_score(data["behavioral_category"], y_pred, average="weighted")
recall    = recall_score(data["behavioral_category"], y_pred, average="weighted")
f1        = f1_score(data["behavioral_category"], y_pred, average="weighted")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

--- Converting to Gravity Units & Calculating ENMO ---
--- Calculating ODBA ---
--- Calculating VeDBA ---
--- Resampling data to 15 second windows ---
Accuracy : 0.6224066390041494
Precision: 0.6035630240664156
Recall   : 0.6224066390041494
F1       : 0.6089631380113205
